<a href="https://colab.research.google.com/github/imthi92/cat-podcast-voice-gen/blob/main/cat_podcast_meow_simba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐱 Cat Podcast - Meow & Simba Voice Generation with VibeVoice

This notebook generates multi-speaker dialogue for the **Cat Podcast Channel** featuring:
- **Meow** - Confident, slightly stupid, thinks he knows everything, makes ridiculous conclusions
- **Simba** - Intelligent, sarcastic, frequently corrects Meow, more practical, delivers punchlines

**Owner:** Mohammed Imthiyaz (imthi92)

**Capabilities**: Long-form multi-speaker TTS (up to 90 min), 4 distinct speakers, natural turn-taking

---

## 🌐 VibeVoice Language Support

| Model | Languages Supported |
|-------|---------------------|
| **VibeVoice-TTS 1.5B** (official) | **English, Chinese** only |
| **VibeVoice-Realtime 0.5B** | English + DE, FR, IT, JP, KR, NL, PL, PT, ES |
| **VibeVoice-ASR** (speech-to-text) | **50+ languages** including Indian languages |
| **Community Hindi fine-tune** | **Hindi** (tarun7r/vibevoice-hindi-1.5B on HuggingFace) |

### Indian Language Support

For TTS (text-to-speech), VibeVoice **officially supports only English and Chinese**.

Indian language options:
- **Hindi**: Community fine-tuned model available (`tarun7r/vibevoice-hindi-1.5B`)
- **Other Indian languages** (Tamil, Telugu, Bengali, Malayalam, Kannada, Marathi, Gujarati, Punjabi, Odia, Assamese, etc.): **No official or community TTS support yet**
- **ASR (speech recognition)**: VibeVoice-ASR supports 50+ languages including major Indian languages

### Indian Slang Support
- No native Indian slang support in VibeVoice
- English with Indian slang may partially work but is not officially tested
- Hindi + English (Hinglish) script can be tested with the community Hindi model

**Recommendation for Indian audiences:** Write scripts in plain English or use the community Hindi model for Hindi content.

---

## 🔗 Resources
- **Model:** microsoft/VibeVoice-1.5B (Hugging Face)
- **Original Repo:** https://github.com/microsoft/VibeVoice
- **Community Hindi Model:** https://huggingface.co/tarun7r/vibevoice-hindi-1.5B
- **Available Voices:** Alice, Carter, Frank, Maya, Sanuel, Anchen, Bowen, Xinran

---

## ⚙️ Hardware Notes
- **GPU Required:** T4 (free Colab) or better
- **VRAM:** ~7 GB for 1.5B model
- **Attention:** Uses SDPA (not flash_attention_2) for T4 compatibility
- **Output:** Saved to Google Drive for persistence

In [1]:
# 1. Install dependencies and clone repository
print("Installing dependencies and setting up environment...")

# Install ffmpeg for audio processing
!apt-get update -y -qq > /dev/null
!apt-get install -y ffmpeg -qq > /dev/null

# Clone VibeVoice repository (community fork if main is unavailable)
import os
if not os.path.exists('VibeVoice'):
    try:
        !git clone https://github.com/microsoft/VibeVoice.git > /dev/null 2>&1
    except:
        !git clone https://github.com/vibevoice-community/VibeVoice.git > /dev/null 2>&1
%cd VibeVoice

# Install flash-attn and VibeVoice package
!pip install flash-attn --no-build-isolation -qq
!pip install -e . -qq

print("Environment setup complete.")

## Character Definitions & Voice Mapping

We'll map Meow and Simba to the closest available VibeVoice preset voices:

| Character | Personality | Mapped Voice | Reason |
|-----------|-------------|--------------|--------|
| **Meow** | Confident, slightly stupid, thinks he knows everything | **Frank** | Frank has a casual, slightly goofy energy that fits Meow |
| **Simba** | Intelligent, sarcastic, practical, delivers punchlines | **Maya** | Maya has a sharp, articulate tone perfect for Simba's sarcasm |

**Available preset voices:** `Alice`, `Carter`, `Frank`, `Maya`, `Sanuel`, `Anchen`, `Bowen`, `Xinran`

In [2]:
# 2. Define the Cat Podcast Episode Script

# ============================================================
# EPISODE: "Why Do Humans Work So Much?" (Pilot Episode)
# Characters: Meow & Simba
# ============================================================

script_text = """
Meow: Humans are strange. They spend eight hours working and then spend another two hours watching cats on the internet.
Simba: You're describing your entire business model, Meow.
Meow: ...I don't see the problem. We're providing a vital service. Entertainment. Education. Emotional support.
Simba: You knock over a plant pot and call it "abstract expressionism."
Meow: It *was* abstract expressionism. The humans just don't understand high art.
Simba: The plant was a succulent. It cost twelve dollars.
Meow: Art has no price, Simba. Besides, I heard the manager say "budget cuts" yesterday. That means more treats for us, right?
Simba: Budget cuts means *fewer* treats, you absolute walnut. It means the humans are stressed. Stressed humans work longer. Longer work means less lap time.
Meow: Wait. So... humans work... to buy treats... but working makes them have less time to give treats?
Simba: Finally, the neuron fires. Welcome to the paradox of capitalism.
Meow: That sounds made up. I prefer my theory: humans are just really bad at being cats.
Simba: *sighs* They're not *trying* to be cats, Meow.
Meow: Well they should try harder. We have it figured out. Sleep sixteen hours. Eat. Sleep. Judge everyone. Repeat.
Simba: You forgot "scream at 3 AM for no reason."
Meow: That's *communication*, Simba. The humans are just bad listeners.
Simba: The manager laughed at his own joke in the meeting today. For five minutes. Five minutes, Meow.
Meow: See? Even the boss knows he's funny. Confidence. That's the key.
Simba: It was a joke about quarterly projections. Nobody laughed but him.
Meow: Exactly! He's his own biggest fan. We should learn from him. From now on, I'm going to laugh at my own jokes.
Simba: You already do.
Meow: *laughs loudly* HA! See? I'm already a successful manager.
Simba: You're a cat who knocked a pen off the desk and called it "strategic reorganization."
Meow: It *was* strategic. The pen was blocking my sunbeam.
Simba: Of course it was. Anyway, back to the original point. Humans work because they've convinced themselves that productivity equals worth.
Meow: But *we* produce nothing. And we're treated like royalty.
Simba: Because we're cute. It's a different economy.
Meow: The cute economy. I like it. We should write a book. "The Cute Economy: How to Get Free Food and Infinite Naps."
Simba: You can't write.
Meow: You can write it. I'll provide the wisdom. Fifty-fifty split.
Simba: Ninety-ten. I do the writing, you provide the "wisdom."
Meow: Eighty-twenty. Final offer.
Simba: *pause* Seventy-thirty. And you have to stop knocking over my water glass.
Meow: ...Deal. But only because I respect the negotiation.
Simba: You don't even know what negotiation means.
Meow: It means I get what I want. Which is... *yawns* ...a nap. Good talk, Simba. Same time tomorrow?
Simba: Same time tomorrow. Don't forget the tuna negotiation.
Meow: *already asleep* Zzz...
Simba: ...Unbelievable. *purrs quietly* Goodnight, you idiot.
"""

In [3]:
# 3. Voice Mapping Configuration

# Map script speakers to VibeVoice preset voices
speaker_voice_mapping = {
    "Meow": "Frank",      # Confident, slightly goofy
    "Simba": "Maya"       # Sharp, articulate, sarcastic
}

## Audio Generation

Now we'll load the model and generate the multi-speaker dialogue.

In [4]:
import os
import re
import torch
import time
import ast
from IPython.display import Audio, display
from google.colab import files

# Import VibeVoice components
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
from transformers.utils import logging

logging.set_verbosity_info()
logger = logging.get_logger(__name__)

# --- Helper Class for Voice Mapping ---
class VoiceMapper:
    def __init__(self):
        self.setup_voice_presets()
        new_dict = {}
        for name, path in self.voice_presets.items():
            if '_' in name: name = name.split('_')[0]
            if '-' in name: name = name.split('-')[-1]
            new_dict[name] = path
        self.voice_presets.update(new_dict)

    def setup_voice_presets(self):
        voices_dir = "demo/voices"
        if not os.path.isdir(voices_dir):
            self.voice_presets, self.available_voices = {}, {}
            print(f"Error: Voices directory not found at path: {os.path.abspath(voices_dir)}")
            return

        wav_files = [f for f in os.listdir(voices_dir) if f.lower().endswith('.wav')]
        self.voice_presets = {os.path.splitext(f)[0]: os.path.join(voices_dir, f) for f in wav_files}
        self.voice_presets = dict(sorted(self.voice_presets.items()))
        self.available_voices = {n: p for n, p in self.voice_presets.items() if os.path.exists(p)}
        print(f"Found {len(self.available_voices)} voice files. Available voices: {', '.join(self.available_voices.keys())}")

    def get_voice_path(self, speaker_name: str) -> str:
        speaker_lower = speaker_name.lower()
        for preset_name, path in self.voice_presets.items():
            if preset_name.lower() == speaker_lower: return path
        for preset_name, path in self.voice_presets.items():
            if speaker_lower in preset_name.lower(): return path
        default_voice = list(self.voice_presets.values())[0]
        print(f"Warning: No voice preset found for '{speaker_name}', using default: {os.path.basename(default_voice)}")
        return default_voice

# --- Main Generation ---
model_path = "microsoft/VibeVoice-1.5B"
output_filename = "cat_podcast_meow_simba_ep01.wav"

try:
    speaker_voice_map = speaker_voice_mapping
    full_script = script_text.strip()

    # Detect unique speakers from script
    unique_speakers_in_script = sorted(list(set(re.findall(r"^(.+?):", full_script, re.MULTILINE))))
    if not unique_speakers_in_script:
        raise ValueError("No speakers found in script. Ensure 'Speaker Name: Text' format.")
    print(f"Detected speakers: {', '.join(unique_speakers_in_script)}")

    # Map speakers to voice files
    voice_mapper = VoiceMapper()
    if not voice_mapper.available_voices:
        raise FileNotFoundError("Could not find voice files in demo/voices/")

    voice_samples = []
    print("Mapping speakers to voices:")
    for speaker in unique_speakers_in_script:
        voice_name = speaker_voice_map.get(speaker)
        if not voice_name:
            raise ValueError(f"Speaker '{speaker}' not in speaker_voice_mapping.")
        voice_path = voice_mapper.get_voice_path(voice_name)
        voice_samples.append(voice_path)
        print(f"  '{speaker}' -> '{voice_name}' ({os.path.basename(voice_path)})")

    # Load processor and model
    print("Loading processor and model (may take 1-2 minutes)...")
    processor = VibeVoiceProcessor.from_pretrained(model_path)
    model = VibeVoiceForConditionalGenerationInference.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map='cuda',
        attn_implementation="sdpa"
    )
    model.eval()
    model.set_ddpm_inference_steps(num_steps=10)
    print("Model loaded.")

    # Prepare inputs
    inputs = processor(
        text=[full_script],
        voice_samples=[voice_samples],
        padding=True,
        return_tensors="pt",
        return_attention_mask=True,
    ).to('cuda')

    # Generate audio
    print("Generating audio (may take several minutes for long dialogue)...")
    start_time = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=None,
        cfg_scale=1.3,
        tokenizer=processor.tokenizer,
        generation_config={'do_sample': False},
        verbose=True,
    )
    generation_time = time.time() - start_time
    print(f"Generation finished in {generation_time:.2f} seconds.")

    # Save and display
    processor.save_audio(outputs.speech_outputs[0], output_filename)
    print(f"Audio saved as '{output_filename}'.")
    display(Audio(output_filename, autoplay=True))

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## Save to Google Drive

Mount Google Drive to persist the generated audio file.

In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy output to Drive
import shutil
drive_output_path = "/content/drive/MyDrive/CatPodcast/MeowSimba/"
os.makedirs(drive_output_path, exist_ok=True)

if os.path.exists(output_filename):
    shutil.copy2(output_filename, os.path.join(drive_output_path, output_filename))
    print(f"Copied to Google Drive: {drive_output_path}{output_filename}")
else:
    print(f"Output file not found: {output_filename}")

## Next Steps

1. **Listen to the generated audio** - Check voice quality, speaker differentiation, natural pauses
2. **Iterate on the script** - Adjust dialogue, pacing, character voices
3. **Try different voice mappings** - Experiment with other preset voices
4. **Create more episodes** - Use the same pipeline for new scripts
5. **Connect to n8n** - Once satisfied, build the automation pipeline:
   - n8n -> Cloud GPU (Colab/RunPod) -> VibeVoice -> Audio -> Google Drive
   - Then add video generation (cat animation + subtitles) -> YouTube upload

---

## Episode Ideas for Future Testing
- "Who Is More Intelligent: Cats or AI?"
- "We Tried to Understand Human Dating"
- "Why Humans Buy Things They Don't Need"
- "Can Cats Run a Business?"
- "AI Took Over Our Podcast"
- "Who Ate the Last Tuna?"
- "Humans and Their Weird Phones"
- "The Manager's Motivational Speech"

---

## Indian Language Options

If you want to create Hindi cat podcast episodes in the future:
1. Use the community model: `tarun7r/vibevoice-hindi-1.5B`
2. Write scripts in Hindi (Devanagari script)
3. Voice sample: `hi-Priya_woman.wav` is available for Hindi female voice
4. For other Indian languages (Tamil, Telugu, etc.), alternative TTS solutions may be needed (Bark, Coqui, or commercial APIs)